In [ ]:
from binder_session import launch_binder_and_get_jupyter_url, BinderTerminal, shutdown_session

binder_base_url = "https://binder.intel4coro.de/build/gh"
repo_path = "yxzhan/robot-lobby/main"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url, verbose=False)

In [ ]:
f"{result['url']}robot-lobby/?token={result['token']}"

In [ ]:
term = BinderTerminal(result["url"], result["token"])

In [ ]:
# out, rc = term.run("ls")
# print(f"[exit {rc}]\n{out}")

out, rc = term.run("ls", stream=True, timeout=60)

## Update the model library on the server

`/shared/mjcf_assets` (the shared volume every session mounts) is on its own branch `stage`, which keeps the stages saved there. This merges `origin/main` (new or changed models) into it, then indexes the library again (`vendor.mjs`) and makes the thumbnails of new models; reload the page afterwards. The session has no git identity, so one is set if missing. A merge that conflicts is aborted (`stage` unchanged); the merge commit stays in the session (push it from there if wanted). Needs `term` from above.

In [ ]:
def run(cmd, timeout=300):
    out, rc = term.run(cmd, timeout=timeout)
    print(f"$ {cmd}\n{out.rstrip()}\n[exit {rc}]")
    return out, rc

run("cd /shared/mjcf_assets && git status -sb | head -5")
branch, _ = run("git branch --show-current")
assert branch.strip().splitlines()[-1:] == ["stage"], "not on stage"
run('git config --global user.name >/dev/null || git config --global user.name "binder"; '
    'git config --global user.email >/dev/null || git config --global user.email "binder@robot-lobby.invalid"')
_, rc = run("git fetch origin main", timeout=600)
assert rc == 0, "fetch failed"
run("git log --oneline stage..origin/main")
_, rc = run("git merge --no-edit origin/main | tail -5")
if rc:
    run("git status --short | head -30")
    run("git merge --abort")
    raise RuntimeError("merge conflict: aborted, stage unchanged")
run("git log --oneline -3")

# The page's asset panel: index the library again, thumbnails for new models.
run('cd "${REPO_DIR:-$HOME}" && node scripts/vendor.mjs | tail -3 && python3 scripts/thumbnails.py | tail -2', timeout=900)

## Upload a file into the session

`upload_file` sends a local file through the Jupyter contents API (in chunks, like JupyterLab's upload); the path is relative to the session's Jupyter root directory.

In [ ]:
from binder_session import upload_file

upload_file(result["url"], result["token"], "binder_session.py", "uploaded/binder_session.py")

out, rc = term.run("ls -l uploaded/")
print(out)

In [ ]:
term.close()

In [ ]:
shutdown_session(result["url"], result["token"])